## Parte 1: Sin Grounding

In [ ]:
from dotenv import load_dotenv
import os
from google import genai

# Cargar variables de entorno
load_dotenv()

# Configuración del cliente
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

# Definición del modelo
MODEL_ID = "gemini-2.5-flash-lite"

def sin_grounding(pregunta):
    try:
        # Llamada correcta a Gemini
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta
        )

        print("\n--- RESPUESTA SIN GROUNDING ---")
        print(response.text)

    except Exception as e:
        print(f"Error: {e}")

sin_grounding("¿Cuál fue el resultado del último partido del Real Madrid?")


--- RESPUESTA SIN GROUNDING ---
Para poder decirte el resultado del último partido del Real Madrid, necesito saber **cuándo** fue ese "último partido". Los resultados de los partidos cambian constantemente.

Por favor, dime la **fecha aproximada** en la que quieres saber el resultado, o si te refieres a un partido específico (por ejemplo, "el último partido de liga", "el último partido de Champions League").


## Parte 2: Implementación del "Buscador Verificado"

In [24]:
def buscador_verificado(pregunta):
    try:
        # Convertimos el formato "messages" a un prompt único
        prompt = f"""
Eres un verificador de noticias profesional.
Responde con información actualizada e indica SIEMPRE las fuentes.

Al final de tu respuesta, incluye una sección llamada FUENTES con el formato:
[1] Título de la fuente - https://url.com
[2] Título de la fuente - https://url.com

Pregunta: {pregunta}
"""

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )

        print("\n--- RESPUESTA VERIFICADA ---")
        print(response.text)

    except Exception as e:
        print(f"Error: {e}")


# Solicitar pregunta al usuario
pregunta_usuario = input("\nIntroduce tu noticia o duda de actualidad: ")
buscador_verificado(pregunta_usuario)


--- RESPUESTA VERIFICADA ---
El último partido disputado por el Real Madrid fue contra el **FC Barcelona**.

El resultado fue una victoria para el Real Madrid por **3-2**.

**Goles:**
*   FC Barcelona: Christensen (minuto 3), Fermín López (minuto 69)
*   Real Madrid: Vinícius Júnior (minuto 18, de penalti), Lucas Vázquez (minuto 73), Bellingham (minuto 90+1)

Este partido se jugó el **domingo, 21 de abril de 2024**, correspondiente a la Jornada 32 de LaLiga.

---

FUENTES:
[1] Real Madrid gana el Clásico y da un golpe a LaLiga - Real Madrid C.F. - https://www.realmadrid.com/noticias/2024/04/real-madrid-gana-el-clasico-y-da-un-golpe-laliga
[2] Real Madrid 3-2 Barcelona (21 Abr, 2024) - ESPN - https://www.espn.com/soccer/match/_/gameId/670831


## Parte 3: El Reto "Anti-Alucinación"

In [25]:
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

def buscador_anti_alucinacion(pregunta):
    try:
        prompt = f"""
Eres un verificador de noticias profesional.
Responde con información actualizada e indica SIEMPRE las fuentes.

Al final de tu respuesta, incluye una sección llamada FUENTES con el formato:
[1] Título de la fuente - https://url.com

Si no tienes fuentes externas reales, escribe exactamente:
FUENTES: NINGUNA

Pregunta: {pregunta}
"""

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            # Activamos grounding para reducir alucinaciones
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]
            )
        )

        texto = response.text

        print("\n--- RESPUESTA ---")

        # Detectar si hay fuentes reales o no
        if "FUENTES: NINGUNA" in texto or "FUENTES:" not in texto:
            print(texto)
            print("\nAdvertencia: Esta respuesta puede no estar verificada en tiempo real")
        else:
            print(texto)

    except Exception as e:
        print(f"Error: {e}")


# Solicitar pregunta al usuario
pregunta_usuario = input("\nIntroduce tu noticia o duda de actualidad: ")
buscador_anti_alucinacion(pregunta_usuario)


--- RESPUESTA ---
El último partido disputado por el Real Madrid fue contra el Bayern de Múnich en la UEFA Champions League, el cual terminó con un resultado de 4-3 a favor del Bayern de Múnich.

FUENTES:
 Real Madrid: partidos en vivo y últimos resultados - 365Scores - https://www.365scores.com/es/futbol/espana/real-madrid
 Real Madrid - Últimos partidos, puntajes y próximos encuentros - FotMob - https://www.fotmob.com/es/teams/7679/calendario/real-madrid/partidos
 Calendario y Próximos Partidos del Real Madrid - LALIGA - https://www.laliga.com/resultado/real-madrid
 Calendario de Partidos Real Madrid en el Santiago Bernabéu - Real Madrid - https://www.realmadrid.com/futbol/masculino/partidos-y-resultados/calendario-de-partidos
 Guía de partidos televisados de Real Madrid - Fútbol en la TV - https://www.futbolenlatele.com/equipos/real-madrid
 Calendario y Próximos Partidos del Real Madrid CF - LALIGA - https://www.laliga.com/ Clubs/real-madrid
 Marcador en vivo, calendario y estadíst